In [1]:
import logging
import sys
from collections.abc import Mapping
from pathlib import Path
from typing import Any

from pydantic import BaseModel, ConfigDict

from wags_llm.client.bedrock import BedrockClaudeJsonClient, EffortLevel
from wags_llm.registry.base import Registry
from wags_llm.services.structured_task import StructuredTaskRunner
from wags_llm.templates.skill_template import SkillTemplate

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.WARNING)

# Effort Demo
Set effort parameter to do some prompt task. Description description

In [2]:
class VariantCurationSkill(SkillTemplate):
    skill_path = Path("skills/variant_curation_0.1.0.md")

    def build_user_prompt(self, payload: Mapping[str, Any]) -> str:
        variant = payload["variant"]
        disease = payload.get("disease", "cancer")

        return f"""Curate the following variant for {disease}.
Variant:
{variant}

Return concise JSON matching the provided schema:
- clinical_significance: one short sentence
- evidence_level: short label
- supporting_rationale: 2-3 short sentences maximum
"""


class VariantCurationResult(BaseModel):
    model_config = ConfigDict(extra="forbid", use_enum_values=True)  # Required

    clinical_significance: str | None = None
    evidence_level: str | None = None
    supporting_rationale: str | None = None
    error_message: str | None = None


skill = VariantCurationSkill()
skill.name, skill.version

('variant_curation', '0.1.0')

In [3]:
registry = Registry()
registry.register(skill)

In [4]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"

In [5]:
skill_params = {
    "skill_name": "variant_curation",
    "skill_version": "0.1.0",
    "payload": {
        "variant": "BRCA2 p.K3326*",
        "disease": "hereditary breast cancer",
    },
    "response_model": VariantCurationResult,
}

In [6]:
# LOW
llm_client_low_effort = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=350,
    temperature=1,
    effort=EffortLevel.LOW,  # LOW, MEDIUM, HIGH, MAX
)

task_runner_low_effort = StructuredTaskRunner(
    client=llm_client_low_effort,
    registry=registry,
)

low_effort_result = task_runner_low_effort.execute_skill(**skill_params)
low_effort_result

VariantCurationResult(clinical_significance='BRCA2 p.K3326* is a recurrent truncating variant considered benign or likely benign for hereditary breast and ovarian cancer risk.', evidence_level='B/LB (ClinVar consensus; population-level evidence)', supporting_rationale='p.K3326* truncates only the last 93 amino acids of BRCA2, a region dispensable for homologous recombination repair. Large case-control studies and functional data show it does not confer significant BRCA2 loss-of-function. ClinVar classifies it as Benign/Likely Benign, though some studies suggest modest risk association with other cancers (e.g., lung, pancreatic), which is not established for hereditary breast cancer.', error_message=None)

In [7]:
# MEDIUM
llm_client_medium_effort = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=700,
    temperature=1,
    effort=EffortLevel.MEDIUM,  # LOW, MEDIUM, HIGH, MAX
)

task_runner_medium_effort = StructuredTaskRunner(
    client=llm_client_medium_effort,
    registry=registry,
)

medium_effort_result = task_runner_medium_effort.execute_skill(**skill_params)
medium_effort_result

VariantCurationResult(clinical_significance='BRCA2 p.K3326* is classified as Benign/Likely Benign for hereditary breast cancer risk.', evidence_level='B (Strong population and functional evidence)', supporting_rationale='This truncating variant affects only the last 93 amino acids of BRCA2 and population studies show it occurs at relatively high frequency (allele frequency ~1%) inconsistent with a high-penetrance pathogenic variant. Large case-control studies (e.g., BRIDGES consortium) have not demonstrated a clinically significant increase in breast cancer risk comparable to known pathogenic BRCA2 truncations. ClinVar and expert panels (ENIGMA) have designated this variant benign or likely benign, though some data suggest a very modest population-level association that does not meet clinical actionability thresholds.', error_message=None)

In [8]:
# HIGH
llm_client_high_effort = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=1400,
    temperature=1,
    effort=EffortLevel.HIGH,  # LOW, MEDIUM, HIGH, MAX
)

task_runner_high_effort = StructuredTaskRunner(
    client=llm_client_high_effort,
    registry=registry,
)

high_effort_result = task_runner_high_effort.execute_skill(**skill_params)
high_effort_result

VariantCurationResult(clinical_significance='BRCA2 p.K3326* is classified as Likely Benign for hereditary breast cancer, as it does not confer clinically actionable increased risk.', evidence_level='Likely Benign (B/LB) — ENIGMA/ClinVar consensus, population-level evidence', supporting_rationale='This nonsense variant truncates only the terminal 7 amino acids of BRCA2 (full length 3418 aa), leaving all critical functional domains—including the RAD51-binding and DNA repair domains—intact. Its minor allele frequency (~0.5–1% in European populations) is far higher than expected for a pathogenic BRCA2 variant, arguing against strong disease causality. The ENIGMA consortium and the majority of ClinVar submitters classify this variant as Benign/Likely Benign for hereditary breast and ovarian cancer, though modest associations with other cancer types (e.g., lung, pancreatic) remain under investigation.', error_message=None)

In [9]:
# MAX
llm_client_max_effort = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=2800,
    temperature=1,
    effort=EffortLevel.MAX,  # LOW, MEDIUM, HIGH, MAX
)

task_runner_max_effort = StructuredTaskRunner(
    client=llm_client_max_effort,
    registry=registry,
)

max_effort_result = task_runner_max_effort.execute_skill(**skill_params)
max_effort_result

VariantCurationResult(clinical_significance='BRCA2 p.K3326* is classified as benign for hereditary breast and ovarian cancer and does not confer meaningful increased BRCA2-associated breast cancer risk.', evidence_level='Benign – Strong (Level A / ENIGMA-validated)', supporting_rationale='The truncation occurs near the extreme C-terminus (codon 3326 of 3418), leaving all critical functional domains—including the RAD51-binding and DNA-binding domains—intact. Large case-control studies and the ENIGMA consortium have shown no significant association with hereditary breast cancer risk, and population allele frequency is notably higher than for pathogenic BRCA2 variants. ClinVar reflects broad consensus classification as benign for HBOC syndrome.', error_message=None)